# Transfer `QwenVL-2.5`-like into SubagentVL Model.

We give an example cookbook to build a `Qwen2.5-VL-7B-Instruct` model into a Self-Calling Agent at inference time. 

In [4]:
from PIL import Image

from qwen_agent.agents import Assistant
# from qwen_agent.utils.output_beautify import typewriter_print, multimodal_typewriter_print
from output_beautify import typewriter_print, multimodal_typewriter_print

from qwen_vl_utils import smart_resize
from tools import encode_pil_image_to_base64, QwenImageVLMTool

## Prepare Image Processing Utils

The Qwen-Agent project is mainly developed for Qwen3-VL. As we 

In [5]:
# for qwen2.5-vl, the factor is 28
FACTOR = 28
MIN_PIXELS = 4 * FACTOR * FACTOR
MAX_PIXELS = 16384 * FACTOR * FACTOR

In [6]:
def image_preprocessing(image, factor=FACTOR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS):
    if isinstance(image, str):
        image = Image.open(image)
    
    assert isinstance(image, Image.Image), "Image must be a PIL image"
    
    width, height = image.size
    input_width, input_height = smart_resize(width, height, factor=28, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    image = image.resize((input_width, input_height))
    return image

## Configure the LLM and the tool for function calling.

You need to serve the LLM using inference like vLLM or SGLang first. For example,

```bash
vllm serve "LOCAL-DIR-FOR-MODEL-CKPT" \
    --port 18902 \
    --gpu-memory-utilization 0.8 \
    --max-model-len 32768 \
    --tensor-parallel-size 1 \
    --served-model-name "MODEL-NAME" \
    --trust-remote-code \
    --disable-log-requests \
    --dtype bfloat16
```

In [ ]:
MODEL_ID = None # model name served by the model service
API_URL = None # base_url, also known as api_base
API_KEY = None # **fill your api key here** if configured

In [ ]:
llm_cfg = {
    # Use dashscope API
    # 'model': 'qwen3-vl-plus',
    # 'model_type': 'qwenvl_dashscope',
    # 'api_key': '' # **fill your api key here**

    # Use a model service compatible with the OpenAI API, such as vLLM or Ollama:
    'model_type': 'qwenvl_oai',
    'model': MODEL_ID,
    'model_server': API_URL,  # base_url, also known as api_base
    'api_key': API_KEY,
    'generate_cfg': {
        "top_p": 0.8,
        "top_k": 40,
        "temperature": 1.0,
        "repetition_penalty": 1.0,
        "presence_penalty": 1.5,
        "stop": ["<|im_end|>\n".strip()],
    }
}


qwen_image_vlm_tool = QwenImageVLMTool()
qwen_image_vlm_tool.register_api({
    "api_url": API_URL,
    "model_id": MODEL_ID,
    "api_key": API_KEY
})

In [ ]:
analysis_prompt = """You are a helpful assistant that can break down tasks into atomic subtasks and assign the subtasks to the **vlm_subagent_tool** if needed.


You can make the subagent to do atomic subtasks like ocr, caption, grounding, subregion-ocr, subregion-caption, subregion-question-answering, subtask-reasoning, etc.
Don't send the user's question to the subagent directly.

Think in <think>...</think> XML tag: Describe the question and image first. Break your task into atomic subtasks. 

Assign the subtasks to the **vlm_subagent_tool** if needed by using <tool_call>...</tool_call> XML tag. 

Aggregate the results and give the final answer in <answer>...</answer> XML tag."""

tools = [
    qwen_image_vlm_tool, 
]
agent = Assistant(
    llm=llm_cfg,
    function_list=tools,
    system_message=analysis_prompt,
    # [!Optional] We provide `analysis_prompt` to enable VL conduct deep analysis. Otherwise use system_message='' to simply enable the tools.
)

Finally, inference time!

In [ ]:
image = None# local or remote image url

image = image_preprocessing(image)
image_url = f"data:image/jpeg;base64,{encode_pil_image_to_base64(image)}"
image.resize((128, 128)).show() # this is just for visualization

question = "Describe the image in detail." # give a hard question to the agent!

messages = []
messages += [
    {"role": "user", "content": [
        {"image": image_url},
        {"text": question}
    ]}
]

response_plain_text = ''
for ret_messages in agent.run(messages):
    # `ret_messages` will contain all subsequent messages, consisting of interleaved assistant messages and tool responses
    response_plain_text = multimodal_typewriter_print(ret_messages, response_plain_text)